given polish_dataset_7_no_chain/merged_is_only


In [85]:
chunk_ids = [
    "classified_information_588",
    "classified_information_595",
    "classified_information_755",
    "education_law_1055",
    "education_law_1522",
    "education_law_1962",
    "education_law_892",
    "financing_education_1008",
    "financing_education_477",
    "financing_education_523",
    "financing_education_652",
    "financing_education_950",
    "healthcare_3004",
    "healthcare_3946",
    "highereducation_science_1331",
    "military_2037",
    "military_2166",
    "police_1208"
]
# chunk that were clearly tt


In [80]:

chunk_ids = [
    "classified_information_623",
    "education_law_1055",
    "education_law_1522",
    "financing_education_652",
    "financing_education_950",
    "healthcare_3946",
    "highereducation_science_1331",
    "highereducation_science_1507",
    "highereducation_science_793",
    "military_1582",
    "military_2037",
    "military_488",
    "police_1208",
    "police_1219",

]

In [86]:
import os
import json
from tqdm import tqdm
import pandas as pd
from FlagEmbedding import BGEM3FlagModel
import numpy as np

# Path to queries.jsonl
queries_path = '../data/processed/polish_dataset_7_no_chain/merged_is_only/queries.jsonl'

# Provided chunk_ids
chunk_ids_set = set(chunk_ids)

# Load queries
queries = []
with open(queries_path, 'r', encoding='utf-8') as f:
    for line in f:
        entry = json.loads(line)
        if entry.get('chunk_id') in chunk_ids_set:
            queries.append(entry)

print(f"Loaded {len(queries)} queries with matching chunk_ids.")

# Inspect a sample
for q in queries[:2]:
    print(q['chunk_id'], q.get('chunk', '')[:60], q.get('context_chunks', '')[:60])


Loaded 18 queries with matching chunk_ids.
financing_education_652 11. Rozliczenie wykorzystania dotacji celowej, o której mowa Art. 62. 1. Na sfinansowanie kosztu zakupu podręczników, mat
military_2166 4. Wezwaniu, o którym mowa w ust. 1, nadaje się rygor natych Art. 211. 1. Posiadacz nieruchomości lub rzeczy ruchomej, wo


In [87]:
# Load BGE-M3 model (sparse and dense)
device = 'mps'  # or 'cuda' if available
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, device=device)

# Helper: cosine similarity for dense vectors
def cosine_dense(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2) + 1e-9)

# Helper: cosine similarity for sparse vectors
def sparse_norm(sv: dict) -> float:
    return np.sqrt(sum(w**2 for w in sv.values()))
def cosine_sparse(sv1: dict, sv2: dict) -> float:
    dot = sum(sv1[k] * sv2[k] for k in sv1 if k in sv2)
    return dot / (sparse_norm(sv1) * sparse_norm(sv2) + 1e-9)


Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 493447.53it/s]


In [88]:
# Compute similarities for each query
results = []
for q in tqdm(queries):
    chunk_id = q['chunk_id']
    query = q.get('query', '')
    chunk = q.get('chunk', '')
    context = q.get('context_chunks', '')
    if not query or not chunk or not context:
        continue
    # Get both sparse and dense embeddings in one call
    emb_query = model.encode([query], return_dense=True, return_sparse=True, return_colbert_vecs=False)
    emb_chunk = model.encode([chunk], return_dense=True, return_sparse=True, return_colbert_vecs=False)
    emb_context = model.encode([context], return_dense=True, return_sparse=True, return_colbert_vecs=False)
    # Scores between query and chunk
    lexical_overlap_chunk = model.compute_lexical_matching_score(
        emb_query['lexical_weights'][0], emb_chunk['lexical_weights'][0])
    sparse_cosine_chunk = cosine_sparse(
        emb_query['lexical_weights'][0], emb_chunk['lexical_weights'][0])
    dense_cosine_chunk = cosine_dense(
        emb_query['dense_vecs'][0], emb_chunk['dense_vecs'][0])
    # Scores between query and context_chunks
    lexical_overlap_context = model.compute_lexical_matching_score(
        emb_query['lexical_weights'][0], emb_context['lexical_weights'][0])
    sparse_cosine_context = cosine_sparse(
        emb_query['lexical_weights'][0], emb_context['lexical_weights'][0])
    dense_cosine_context = cosine_dense(
        emb_query['dense_vecs'][0], emb_context['dense_vecs'][0])
    results.append({
        'chunk_id': chunk_id,
        'lexical_overlap_chunk': lexical_overlap_chunk,
        'sparse_cosine_chunk': sparse_cosine_chunk,
        'dense_cosine_chunk': dense_cosine_chunk,
        'lexical_overlap_context': lexical_overlap_context,
        'sparse_cosine_context': sparse_cosine_context,
        'dense_cosine_context': dense_cosine_context,
        'query': query,
        'chunk': chunk,
        'context_chunk': context
    })

results_df = pd.DataFrame(results)
results_df.head()


  0%|          | 0/18 [00:00<?, ?it/s]

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
100%|██████████| 18/18 [00:03<00:00,  5.52it/s]
/Users/m-proj/repositories/long-context-retrieval/.venv/lib/python3.11/site-packages/pandas/io/formats/format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,chunk_id,lexical_overlap_chunk,sparse_cosine_chunk,dense_cosine_chunk,lexical_overlap_context,sparse_cosine_context,dense_cosine_context,query,chunk,context_chunk
0,financing_education_652,0.214966,0.428711,0.804688,0.260254,0.387451,0.697754,Czy szkoły artystyczne prowadzone przez podmio...,"11. Rozliczenie wykorzystania dotacji celowej,...",Art. 62. 1. Na sfinansowanie kosztu zakupu pod...
1,military_2166,0.149170,0.334961,0.651367,0.261719,0.376465,0.749512,Czy wezwanie wydane przez wójta lub burmistrza...,"4. Wezwaniu, o którym mowa w ust. 1, nadaje si...",Art. 211. 1. Posiadacz nieruchomości lub rzecz...
2,highereducation_science_1331,0.077576,0.293457,0.463867,0.101318,0.349854,0.486572,Czy Dyrektor NAWA uwierzytelnia duplikaty świa...,"2) duplikaty dokumentów, o których mowa w pkt 1;",1) świadectwa ukończenia studiów podyplomowych;
3,healthcare_3946,0.252930,0.434570,0.733398,0.192017,0.512695,0.604980,Czy w razie stwierdzenia naruszeń minister zdr...,"3. W razie stwierdzenia, na podstawie uzyskany...","3) podmiotów, którym Fundusz powierzył wykonyw..."
4,police_1208,0.242676,0.615723,0.701172,0.064819,0.105286,0.554199,Czy komendant szkoły policyjnej może w każdym ...,"10. Przełożony, o którym mowa w art. 32 ust. 1...",1 67) W brzmieniu ustalonym przez art. 2 pkt 7...


In [89]:
from lcr.config import DATA_DIR
save_dir = DATA_DIR / 'experiments/r4_vs_r6_analysis'


results_df.sort_values('chunk_id').to_csv(save_dir / 'bge_m3_r4_scores.csv', index=False)

In [95]:
r6 = pd.read_csv(save_dir / 'bge_m3_r6_scores.csv')
r4 = pd.read_csv(save_dir / 'bge_m3_r4_scores.csv')
r4r6_inner = pd.merge(r4, r6, on='chunk_id', suffixes=('_r4', '_r6'))
r4r6_inner.head()

,chunk_id,lexical_overlap_chunk_r4,sparse_cosine_chunk_r4,dense_cosine_chunk_r4,lexical_overlap_context_r4,sparse_cosine_context_r4,dense_cosine_context_r4,query_r4,chunk_r4,context_chunk_r4,lexical_overlap_chunk_r6,sparse_cosine_chunk_r6,dense_cosine_chunk_r6,lexical_overlap_context_r6,sparse_cosine_context_r6,dense_cosine_context_r6,query_r6,chunk_r6,context_chunk_r6
0,education_law_1055,0.11993,0.3510,0.6646,0.2163,0.4456,0.6380,Czy nauczyciele zatrudnieni w kuratoriach oświ...,"4. Nauczyciele, o których mowa w art. 60 ust. ...","8. Kurator oświaty oraz organy, o których mowa...",0.04596,0.16210,0.6177,0.187866,0.4666,0.6284,Które organy sprawują nadzór pedagogiczny przy...,"4. Nauczyciele, o których mowa w art. 60 ust. ...","8. Kurator oświaty oraz organy, o których mowa..."
1,education_law_1522,0.09530,0.4243,0.6260,0.2017,0.4358,0.7500,Czy przedszkolny oddział w szkole podstawowej ...,2. Przepisy ust. 1 stosuje się odpowiednio do ...,Art. 102. 1. Statut przedszkola zawiera w szcz...,0.10710,0.58000,0.6200,0.054565,0.1433,0.6416,Jakie elementy musi zawierać statut oddziału p...,2. Przepisy ust. 1 stosuje się odpowiednio do ...,Art. 102. 1. Statut przedszkola zawiera w szcz...
2,financing_education_652,0.21500,0.4287,0.8047,0.2603,0.3875,0.6978,Czy szkoły artystyczne prowadzone przez podmio...,"11. Rozliczenie wykorzystania dotacji celowej,...",Art. 62. 1. Na sfinansowanie kosztu zakupu pod...,0.22080,0.60550,0.7593,0.152466,0.3120,0.6025,"Jaki jest cel dotacji celowej, której rozlicze...","11. Rozliczenie wykorzystania dotacji celowej,...",Art. 62. 1. Na sfinansowanie kosztu zakupu pod...
3,financing_education_950,0.24880,0.5205,0.7134,0.4090,0.4106,0.7190,Czy podręczniki przyznane w ramach art. 119 na...,"3. Podręczniki, o których mowa w ust. 1, są do...",Art. 119. 1. Na lata szkolne 2017/2018 i 2018/...,0.00867,0.01897,0.6210,0.142700,0.1499,0.6700,Which textbooks provided to primary schools fo...,"3. Podręczniki, o których mowa w ust. 1, są do...",Art. 119. 1. Na lata szkolne 2017/2018 i 2018/...
4,healthcare_3946,0.25300,0.4346,0.7334,0.1920,0.5127,0.6050,Czy w razie stwierdzenia naruszeń minister zdr...,"3. W razie stwierdzenia, na podstawie uzyskany...","3) podmiotów, którym Fundusz powierzył wykonyw...",0.25560,0.44290,0.7040,0.217041,0.5840,0.6367,"Jakie podmioty, którym Fundusz powierzył wykon...","3. W razie stwierdzenia, na podstawie uzyskany...","3) podmiotów, którym Fundusz powierzył wykonyw..."


In [96]:
# lexical diff
r4r6_inner[['lexical_overlap_chunk_r4', 'lexical_overlap_context_r6']]

,lexical_overlap_chunk_r4,lexical_overlap_context_r6
0,0.11993,0.187866
1,0.09530,0.054565
2,0.21500,0.152466
3,0.24880,0.142700
4,0.25300,0.217041
5,0.07760,0.000000
6,0.24740,0.173218
7,0.24270,0.038391


In [97]:
# dense diff
r4r6_inner[['dense_cosine_chunk_r4', 'dense_cosine_context_r6']]

,dense_cosine_chunk_r4,dense_cosine_context_r6
0,0.6646,0.6284
1,0.6260,0.6416
2,0.8047,0.6025
3,0.7134,0.6700
4,0.7334,0.6367
5,0.4639,0.4065
6,0.6480,0.6514
7,0.7010,0.4812
